In [ ]:
from typing import Mapping, Sequence
from pathlib import Path

import numpy as np
import polars as pl
import matplotlib.pyplot as plt

import pulp

In [ ]:
# problem description

# We are given some forecasts for demands for our products.
# We want to optimise how much of each product to store at any given point in time 
# in order to maximise profit (maximum revenue and minimum cost).
# We only have limited amount of storage for each product category ()
# and each product carries a penalty cost for storing excess product in the warehouse.

In [ ]:
# Constants

INPUT_BASE_PATH = "../data"
FORECASTS_PATH = Path("../results/ets/submissions")

# ==================
# FORECAST CONSTANTS 
# ==================
MAX_TRAINING_TIMESTAMP = 1941
FORECAST_HORIZON = 28
SUBMISSION_F_COLS = [f"F{i}" for i in range(1, FORECAST_HORIZON + 1)]

# ==================================================
# OPTIMIZATION CONSTANTS / PARAMETERS / CONSTRAINTS
# ==================================================
TOTAL_STORAGE_CAPACITY = 45_000

# Gamma distributed
FOOD_VOLUME_DIST_PARAMS = {"shape": 5, "scale": 10}
HOUSEHOLD_VOLUME_DIST_PARAMS = {"shape": 10, "scale": 15}
HOBBIES_VOLUME_DIST_PARAMS = {"shape": 15, "scale": 17}

# Uniformly distributed
FOOD_STORAGE_COST_DIST_PARAMS = {"low": 0.1, "high": 0.5}
HOUSEHOLD_STORAGE_COST_DIST_PARAMS = {"low": 0.2, "high": 0.3}
HOBBIES_STORAGE_COST_DIST_PARAMS = {"low": 0.3, "high": 0.7}

FORECAST_ID_SUFFIX = "_evaluation"
FOODS_PRODUCT_STORE_IDS = [
    "FOODS_3_555_TX_2",
    "FOODS_3_376_TX_2",
    "FOODS_3_811_CA_2",
    "FOODS_1_218_TX_1",
    "FOODS_3_226_WI_3",
    "FOODS_3_070_WI_2",
    "FOODS_3_007_TX_2",
    "FOODS_3_444_TX_1",
    "FOODS_2_398_WI_3",
    "FOODS_3_540_WI_1",
]
HOUSEHOLD_PRODUCT_STORE_IDS = [
    "HOUSEHOLD_1_334_TX_1",
    "HOUSEHOLD_1_459_CA_2",
    "HOUSEHOLD_2_342_WI_2",
    "HOUSEHOLD_1_465_TX_3",
    "HOUSEHOLD_1_294_WI_2",
    "HOUSEHOLD_2_176_CA_3",
    "HOUSEHOLD_1_334_TX_2",
    "HOUSEHOLD_1_106_WI_2",
    "HOUSEHOLD_1_474_TX_2",
    "HOUSEHOLD_1_106_TX_1",
]
HOBBIES_PRODUCT_STORE_IDS = [
    "HOBBIES_1_048_WI_1",
    "HOBBIES_1_067_CA_3",
    "HOBBIES_1_158_TX_3",
    "HOBBIES_1_404_WI_3",
    "HOBBIES_1_234_CA_3",
    "HOBBIES_1_254_CA_3",
    "HOBBIES_1_019_WI_1",
    "HOBBIES_1_370_WI_1",
    "HOBBIES_1_048_CA_1",
    "HOBBIES_1_354_TX_3",
]
ALL_PRODUCT_STORE_IDS = list(FOODS_PRODUCT_STORE_IDS + HOUSEHOLD_PRODUCT_STORE_IDS + HOBBIES_PRODUCT_STORE_IDS)
ALL_PRODUCT_STRORE_IDS_WITH_SUFFIX = [f"{p}{FORECAST_ID_SUFFIX}" for p in ALL_PRODUCT_STORE_IDS]

RNG = np.random.default_rng(42)

In [ ]:
# Load data and forecasts

SELL_PRICES = pl.read_csv(f"{INPUT_BASE_PATH}/sell_prices.csv")
CALENDAR_DATA = pl.read_csv(f"{INPUT_BASE_PATH}/calendar.csv", try_parse_dates=True)
SALES_TRAIN_EVALUATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_evaluation.csv")
SALES_TRAIN_VALIDATION = pl.read_csv(f"{INPUT_BASE_PATH}/sales_train_validation.csv")
FORECASTS = pl.read_csv(f"{FORECASTS_PATH}/submission_2026-07-12 21:48:22.csv")

In [ ]:
# Prepare forecast df

FORECAST_DF = (
    FORECASTS.filter(pl.col("id").is_in(ALL_PRODUCT_STRORE_IDS_WITH_SUFFIX))
    .unpivot(index="id", value_name="sales", variable_name="F")
    .with_columns(
        item_id=pl.col("id").str.extract(r"^([^_]+_[^_]+_[^_]+)"),
        item_store_id=pl.col("id").str.strip_suffix(FORECAST_ID_SUFFIX),
        dept_id=pl.col("id").str.extract(r"^([^_]+)"),
        F_index=pl.col("F").str.strip_chars_start("F").cast(pl.Int64)
    )
    .with_columns(
        d=pl.format("d_{}", pl.col("F_index") + MAX_TRAINING_TIMESTAMP),
        d_index=pl.col("F_index") + MAX_TRAINING_TIMESTAMP
    )
    .join(
        other=CALENDAR_DATA.select(["date", "wm_yr_wk", "d"]),
        on="d",
        how="left"
    )
    .join(
        other=(
            SELL_PRICES
            .with_columns(item_store_id=pl.format("{}_{}", pl.col("item_id"), pl.col("store_id")))
            .drop(pl.col("item_id"), pl.col("store_id"))
        ),
        on=["item_store_id", "wm_yr_wk"],
        how="left",
    )
    .select(["id", "item_id", "item_store_id", "dept_id", "F", "F_index", "d", "d_index", "sales", "sell_price"])
)

FORECAST_DF

In [ ]:
# Define storage volume per product

food_product_volumes = RNG.gamma(**FOOD_VOLUME_DIST_PARAMS, size=len(FOODS_PRODUCT_STORE_IDS))
food_product_id_to_volume = pl.DataFrame({"item_store_id": FOODS_PRODUCT_STORE_IDS, "volume": food_product_volumes})

household_product_volumes = RNG.gamma(**HOUSEHOLD_VOLUME_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_STORE_IDS))
household_product_id_to_volume = pl.DataFrame({"item_store_id": HOUSEHOLD_PRODUCT_STORE_IDS, "volume": household_product_volumes})

hobbies_product_volumes = RNG.gamma(**HOBBIES_VOLUME_DIST_PARAMS, size=len(HOBBIES_PRODUCT_STORE_IDS))
hobbies_product_id_to_volume = pl.DataFrame({"item_store_id": HOBBIES_PRODUCT_STORE_IDS, "volume": hobbies_product_volumes})

VOLUME_DF = (
    FORECAST_DF
    .select(["id", "item_id", "item_store_id"])
    .unique()
    .join(
        other=food_product_id_to_volume.rename({"volume": "volume_foods"}),
        on="item_store_id",
        how="left"
    ).join(
        other=household_product_id_to_volume.rename({"volume": "volume_household"}),
        on="item_store_id",
        how="left",
    ).join(
        other=hobbies_product_id_to_volume.rename({"volume": "volume_hobbies"}),
        on="item_store_id",
        how="left"
    )
    .with_columns(
        storage_volume=pl.coalesce(pl.selectors.starts_with("volume_"))
    ).drop(
        pl.selectors.starts_with("volume_"),
        pl.col("id"),
        pl.col("item_id"),
    )
)

VOLUME_DF

In [ ]:
# Define storage cost per item as a proportion of average sell
# price for item.

average_sell_prices = (
    SELL_PRICES
    .with_columns(
        item_store_id=pl.format("{}_{}", pl.col("item_id"), pl.col("store_id")),
        dept_id=pl.col("item_id").str.extract(r"^([^_]+)")
    )
    .filter(pl.col("item_store_id").is_in(ALL_PRODUCT_STORE_IDS))
    .group_by(pl.col("item_store_id"), pl.col("dept_id"))
    .agg(avg_sell_price=pl.col("sell_price").mean())
    .sort(pl.col("dept_id"), pl.col("item_store_id"))
)

food_storage_costs_pct = RNG.uniform(**FOOD_STORAGE_COST_DIST_PARAMS, size=len(FOODS_PRODUCT_STORE_IDS))
food_product_id_to_cost_pct = pl.DataFrame({"item_store_id": FOODS_PRODUCT_STORE_IDS, "cost_pct": food_storage_costs_pct})

household_storage_costs_pct = RNG.uniform(**HOUSEHOLD_STORAGE_COST_DIST_PARAMS, size=len(HOUSEHOLD_PRODUCT_STORE_IDS))
household_product_id_to_cost_pct = pl.DataFrame({"item_store_id": HOUSEHOLD_PRODUCT_STORE_IDS, "cost_pct": household_storage_costs_pct})

hobbies_storage_costs_pct = RNG.uniform(**HOBBIES_STORAGE_COST_DIST_PARAMS, size=len(HOBBIES_PRODUCT_STORE_IDS))
hobbies_product_id_to_cost_pct = pl.DataFrame({"item_store_id": HOBBIES_PRODUCT_STORE_IDS, "cost_pct": hobbies_storage_costs_pct})

COST_DF = (
    average_sell_prices.join(
        other=food_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_food"}),
        how="left",
        on="item_store_id",
    ).join(
        other=household_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_household"}),
        how="left",
        on="item_store_id"
    ).join(
        other=hobbies_product_id_to_cost_pct.rename({"cost_pct": "cost_pct_hobbies"}),
        how="left",
        on="item_store_id"
    ).with_columns(
        cost_pct=pl.coalesce(pl.selectors.starts_with("cost_pct_"))
    ).with_columns(
        storage_cost=(pl.col("cost_pct") * pl.col("avg_sell_price")).round(2)
    ).drop(
        pl.selectors.starts_with("cost_pct"),
        pl.col("avg_sell_price"),
        pl.col("dept_id"),
    )
)

COST_DF

In [ ]:
def optimise_single_timestep(
    products: Sequence[str],
    expected_sales: Mapping[str, int],
    sell_prices: Mapping[str, float],
    storage_costs: Mapping[str, float],
    storage_volumes: Mapping[str, float],
) -> dict[str, int] | None:
    # Define new optimisation problem for this timestamp
    inventory_planning = pulp.LpProblem("InventoryPlanning", sense=pulp.LpMinimize)

    # Decision variable
    x = pulp.LpVariable.dict(name="inventory", indices=products, lowBound=0, cat=pulp.LpInteger)

    # Auxiliary variable
    t = pulp.LpVariable.dict(name="t", indices=products, lowBound=0, cat=pulp.LpInteger)

    # Objective
    # This is a piecewise linear convex optimisation problem. Decision variables are optimised 
    # via constraints on the auxiliary variable. Objective is minimising the total sum of all
    # auxiliary variables
    inventory_planning += pulp.lpSum([t[p] for p in products])

    # Constraints
    for p in products:
        # Constraint for missing out on possible sales
        inventory_planning += (expected_sales[p] - x[p]) * sell_prices[p] <= t
        
        # Constraint for storing more than expected demand
        inventory_planning += (x[p] - expected_sales[p]) * storage_costs[p] <= t

    # Storage constraint
    inventory_planning += pulp.lpSum([storage_volumes[p] * x[p] for p in products]) <= TOTAL_STORAGE_CAPACITY

    inventory_planning.solve(pulp.PULP_CBC_CMD(msg=False))
    status = pulp.LpStatus[inventory_planning.status]
    
    if status != "Optimal":
        print(f"No optimal solution found. Solution status {status}")
        return None

    return {p: x[p].varValue for p in products}
    

In [ ]:
# Inventory optimisation

products = list(ALL_PRODUCT_STORE_IDS)
forecast_timestamp = 1

# Convert cost and storage dataframes to dicts for easy access

cost_dict = dict(
    zip(
        COST_DF["item_store_id"].to_list(),
        COST_DF["storage_cost"].to_list(),
    )
)
volume_dict = dict(
    zip(
        VOLUME_DF["item_store_id"].to_list(),
        VOLUME_DF["storage_volume"].to_list(),
    )
)

inventory_dfs: list[pl.DataFrame] = []
for forecast_timestamp in range(1, FORECAST_HORIZON + 1):
    # Filter forecasts to current timestamp
    forecast_df_at_t = FORECAST_DF.filter(pl.col("F_index") == forecast_timestamp)
    assert not forecast_df_at_t.is_empty()
    sales_dict_at_t = dict(
        zip(
            forecast_df_at_t["item_store_id"].to_list(),
            forecast_df_at_t["sales"].to_list(),
        )
    )
    sell_price_dict_at_t = dict(
        zip(
            forecast_df_at_t["item_store_id"].to_list(),
            forecast_df_at_t["sell_price"].to_list(),
        )
    )

    inventory_at_t = optimise_single_timestep(
        products=ALL_PRODUCT_STORE_IDS,
        expected_sales=sales_dict_at_t,
        sell_prices=sell_price_dict_at_t,
        storage_costs=cost_dict,
        storage_volumes=volume_dict,
    )
    
    inventory_at_t_df: pl.DataFrame
    if inventory_at_t is None:
        inventory_at_t_df = (
            pl.DataFrame(
                {
                    "product_id": ALL_PRODUCT_STORE_IDS,
                    "F_index": [forecast_timestamp for _ in range(len(ALL_PRODUCT_STORE_IDS))]
                }
            )
            .with_columns(inventory=pl.lit(float("nan")))
            .select(["product_id", "inventory", "F_index"])
        )
        inventory_dfs.append(inventory_at_t_df)
        continue


    product_ids, inventory = zip(*inventory_at_t.items())
    inventory_at_t_df = pl.DataFrame(
        {
            "product_id": product_ids,
            "inventory": inventory,
            "F_index": [forecast_timestamp for _ in range(len(product_ids))]
        }
    )
    inventory_dfs.append(inventory_at_t_df)

OPTIMAL_INVENTORY_DF = (
    pl.concat(inventory_dfs, how="vertical_relaxed")
    .sort(pl.col("F_index"), pl.col("product_id"))
)

In [ ]:
# Analysis